# S15 · Lab — Nifty 50: ARIMA vs exponential smoothing

The session lab. We take one real series, the Nifty 50 daily index, fit BOTH an
ARIMA model and an exponential-smoothing model on the same data, forecast a hidden
tail, and judge them honestly side by side. This is a proper forecasting bake-off:
same data, same test, no cheating.

**New here? Read this once.**

- New to Python? You can still run this whole lab. Press play on each cell, top to
  bottom, and read the plain-English note above each one.
- New to time series? Open the primer
  `primers/time_series_and_forecasting.md` first.
- Already confident? Look for the cells marked **Stretch (optional)** and the
  GARCH peek near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there. We install `yfinance`
because Colab does not ship it.

In [ ]:
# yfinance is not on Colab by default; statsmodels usually is, installed to be safe.
import sys
if "google.colab" in sys.modules:
    !pip install -q yfinance statsmodels
else:
    print("Not on Colab - assuming the libraries are already installed.")

In [ ]:
import numpy as np                     # fast maths on lists of numbers
import pandas as pd                      # tables and dated series
import matplotlib.pyplot as plt          # drawing charts

## What we are forecasting

The **Nifty 50** is the headline Indian stock-market index: think of it as one
number that summarises the 50 biggest companies on the National Stock Exchange.
Each trading day it has a closing value, and that daily closing value is our time
series.

You do not need any finance background. It is simply a column of numbers with a
date attached, exactly like the made-up demand series in the earlier notebooks.

## Step 1 — get the data (with a safe offline fallback)

We try to download the real Nifty 50 daily series with `yfinance`. If there is no
internet, or the download comes back empty, we quietly fall back to a
**synthetic** series built with numpy (trend + season + a random-walk wobble) so
the notebook always runs. We print which one we ended up using. The seed keeps the
fallback reproducible.

In [ ]:
# Set a seed so the fallback series (and anything random) is reproducible.
np.random.seed(0)

using_live_data = False

try:
    import yfinance as yf
    # "^NSEI" is the ticker symbol for the Nifty 50 index.
    downloaded = yf.download("^NSEI", start="2021-01-01", end="2023-12-31",
                             progress=False, auto_adjust=True)
    prices = downloaded["Close"].dropna()
    # yfinance may return a 1-column table; squeeze it to a plain series.
    prices = prices.squeeze()
    if len(prices) < 100:
        raise ValueError("download was empty or too short")
    using_live_data = True
    print("Downloaded live Nifty 50 data:", len(prices), "trading days.")
except Exception:
    # Offline fallback: a synthetic but realistic-looking price series.
    print("Could not download live data; using a synthetic series instead.")
    number_of_days = 740
    day = np.arange(number_of_days)
    trend = 14000 + 6.0 * day                                     # slow upward drift
    season = 300 * np.sin(2 * np.pi * day / 250)                  # a gentle yearly swing
    wobble = np.cumsum(np.random.normal(0, 40, number_of_days))   # random-walk wobble
    synthetic_values = trend + season + wobble
    dates = pd.bdate_range("2021-01-01", periods=number_of_days)  # business days only
    prices = pd.Series(synthetic_values, index=dates)

print("Live data used?", using_live_data)
print(prices.head())

## Step 2 — plot the price series

Always look at the data first. Whether it is live or synthetic, you should see a
long drifting line, the kind of series that trends and does not sit at one level.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(prices.index, prices.values, color="#2E75B6")
plt.xlabel("date")
plt.ylabel("Nifty 50 closing value")
plt.title("Nifty 50 daily closing value")
plt.show()

## Step 3 — hold out the last part for an honest test

The golden rule of forecasting: judge a model on data it has **never seen**. So we
hide the last 30 trading days. We train both models on the earlier part, then
compare their forecasts against that hidden tail. This is the same idea as a
train/test split, but for data that arrives in time order we always hide the
*most recent* stretch.

In [ ]:
# How many days to hide at the end.
hold_out_size = 30

# Everything except the last 30 days is for training.
training_series = prices.iloc[:-hold_out_size]

# The last 30 days are the hidden truth we will score against.
held_out_series = prices.iloc[-hold_out_size:]

print("training days:", len(training_series))
print("held-out days:", len(held_out_series))

## Step 4 — check stationarity and pick the differencing order d

Stock prices trend, so they are not stationary. We run the **ADF test** on the
prices and on their day-to-day changes. A small p-value (below 0.05) means
stationary. The changes are usually stationary while the prices are not, which
tells us to difference once, so `d = 1` in ARIMA.

In [ ]:
from statsmodels.tsa.stattools import adfuller

price_changes = training_series.diff().dropna()

adf_on_prices = adfuller(training_series)
adf_on_changes = adfuller(price_changes)

print("ADF p-value on prices       :", round(adf_on_prices[1], 3))
print("ADF p-value on daily changes:", round(adf_on_changes[1], 3))
print()
print("Prices are not stationary; their changes are. So we use d = 1.")

## Step 5 — fit an ARIMA model

We fit a small, fast ARIMA(1, 1, 1) on the training part: difference once
(`d = 1`), lean on one past value (`p = 1`) and one past shock (`q = 1`). A daily
index behaves almost like a random walk, where the best guess for tomorrow is
close to today, so we do not expect miracles. But it makes an honest baseline.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

arima_model = ARIMA(training_series, order=(1, 1, 1))
arima_fit = arima_model.fit()

# Forecast the same number of days we held out, and ask for a band too.
arima_forecast_object = arima_fit.get_forecast(steps=hold_out_size)
arima_forecast = arima_forecast_object.predicted_mean
arima_band = arima_forecast_object.conf_int(alpha=0.05)

print("ARIMA forecast, first 5 days:")
print(arima_forecast.head())

## Step 6 — quickly check the ARIMA residuals

A fast look at the leftovers. The **Ljung-Box** p-value should ideally be above
0.05, meaning the residuals look like noise. For near-random-walk data like a
stock index it often is, which is a sign the simple model is reasonable.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

residual_test = acorr_ljungbox(arima_fit.resid, lags=[10], return_df=True)
print(residual_test)
print()
print("lb_pvalue above 0.05 means the residuals look like noise - good.")

## Step 7 — fit an exponential-smoothing model

On the very same training series we fit Holt's method (level + trend). A daily
index has a drift but no clean repeating season, so Holt is the natural
exponential-smoothing choice here. Picking the method to match the shape of the
data is the whole lesson from notebook 02.

In [ ]:
from statsmodels.tsa.holtwinters import Holt

exp_smoothing_fit = Holt(training_series).fit()

# Forecast the same hold-out horizon.
exp_smoothing_forecast = exp_smoothing_fit.forecast(hold_out_size)

# Give the forecast the same dates as the held-out tail so they line up on plots.
exp_smoothing_forecast.index = held_out_series.index

print("Exponential-smoothing forecast, first 5 days:")
print(exp_smoothing_forecast.head())

## Step 8 — score both models on the hidden tail

Now the honest comparison. For each model we measure the **root mean squared
error** (RMSE) against the held-out truth: the typical size of the miss, in index
points. Smaller is better.

In [ ]:
# Root mean squared error: square the misses, average them, take the square root.
def root_mean_squared_error(actual, predicted):
    difference = np.asarray(actual) - np.asarray(predicted)
    return np.sqrt(np.mean(difference ** 2))

arima_rmse = root_mean_squared_error(held_out_series.values, arima_forecast.values)
exp_smoothing_rmse = root_mean_squared_error(held_out_series.values,
                                             exp_smoothing_forecast.values)

print("RMSE on the hidden 30 days (smaller is better):")
print("  ARIMA(1,1,1)         :", round(arima_rmse, 1))
print("  Exponential smoothing:", round(exp_smoothing_rmse, 1))
print()
if arima_rmse < exp_smoothing_rmse:
    print("ARIMA was closer on this hold-out.")
else:
    print("Exponential smoothing was closer on this hold-out.")

## Step 9 — plot both forecasts against the truth

We zoom into the end of the series so the comparison is clear: the recent history,
the hidden truth, and the two forecasts. The shaded band is ARIMA's 95% prediction
interval. Remember, a forecast is a fan, not a line.

In [ ]:
plt.figure(figsize=(11, 5))

# Show just the last stretch of training history so the picture is not cramped.
recent_history = training_series.iloc[-90:]
plt.plot(recent_history.index, recent_history.values,
         color="#2E75B6", label="history")

# The hidden truth we scored against.
plt.plot(held_out_series.index, held_out_series.values,
         color="black", linewidth=2, label="actual (held out)")

# The two forecasts.
plt.plot(arima_forecast.index, arima_forecast.values,
         color="#C0392B", linestyle="--", label="ARIMA forecast")
plt.plot(exp_smoothing_forecast.index, exp_smoothing_forecast.values,
         color="#E67E22", linestyle="--", label="exp. smoothing forecast")

# ARIMA's prediction interval (a lower and an upper column).
arima_lower = arima_band.iloc[:, 0]
arima_upper = arima_band.iloc[:, 1]
plt.fill_between(arima_forecast.index, arima_lower, arima_upper,
                 color="#C0392B", alpha=0.15, label="ARIMA 95% interval")

plt.xlabel("date")
plt.ylabel("Nifty 50 closing value")
plt.title("ARIMA vs exponential smoothing on held-out Nifty 50 data")
plt.legend()
plt.show()

## What you just did

You ran a real forecasting bake-off: two classical models on the same series,
scored honestly on days they never saw, with a prediction interval drawn around
the forecast. The usual lesson: a daily stock index is close to a random walk, so
both models give modest, wide forecasts, and the simple exponential-smoothing
model is often surprisingly hard to beat. Both are baselines worth beating before
reaching for anything fancier.

### Stretch (optional, no code) — a 5-minute peek at GARCH

Everything above forecasts *where* the series is heading. There is a different
question a bank or a trader cares about just as much: *how wild* will it be? Look
back at the plot and you will notice the wobble is not the same size everywhere:
calm stretches and stormy stretches cluster together. That changing size of the
wobble is called **volatility**, and ARIMA cannot capture it, because ARIMA
assumes the noise stays the same size.

The standard tool for changing volatility is **GARCH**. The one-line difference to
remember: ARIMA models the **mean** of a series (where it will be), while GARCH
models its changing **variance** (how wild it will be). It is the workhorse for
risk and option pricing in finance.

You do not build one today. Just know the door exists. In the `arch` package the
API looks like this (a sketch, not to run here):

```python
from arch import arch_model

returns = prices.pct_change().dropna() * 100   # daily returns, in percent
garch = arch_model(returns, vol="Garch", p=1, q=1)
garch_fit = garch.fit(disp="off")
print(garch_fit.summary())
```

A full treatment belongs to a dedicated quant-finance course. For today, the takeaway
is only that the word exists and what gap it fills. The `arch` docs are the place
to read more.